# RecKAN on MNIST — v2 (train/val split)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# =========================
# 1. Recursive polynomial basis
# =========================

def recursive_poly_basis(x, degree, a_raw, b_raw, c_raw, d_raw, e_raw, param_bound=3.0):
    a = param_bound * torch.tanh(a_raw)
    b = param_bound * torch.tanh(b_raw)
    c = param_bound * torch.tanh(c_raw)
    d = param_bound * torch.tanh(d_raw)
    e = param_bound * torch.tanh(e_raw)

    x = x.clamp(-2.0, 2.0)

    Rm1 = torch.zeros_like(x)   # R_0
    R0 = torch.ones_like(x)     # R_1

    polys = [Rm1, R0]

    for _ in range(1, degree):
        coef1 = a * x**2 + b * x + c
        coef2 = d * x + e
        R1 = coef1 * R0 + coef2 * Rm1
        polys.append(R1)
        Rm1, R0 = R0, R1

    return torch.stack(polys, dim=-1)


class RecursivePolyKANLayer(nn.Module):
    def __init__(self, input_dim, output_dim, degree=3, param_bound=3.0):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.degree = degree
        self.param_bound = param_bound

        self.weights = nn.Parameter(
            torch.randn(output_dim, input_dim, degree + 1) * 0.02
        )

        self.a_raw = nn.Parameter(torch.tensor(0.0))
        self.b_raw = nn.Parameter(torch.tensor(0.0))
        self.c_raw = nn.Parameter(torch.tensor(0.0))
        self.d_raw = nn.Parameter(torch.tensor(0.0))
        self.e_raw = nn.Parameter(torch.tensor(0.0))

        with torch.no_grad():
            self.a_raw.fill_(0.0)
            self.b_raw.fill_(0.5)
            self.c_raw.fill_(0.0)
            self.d_raw.fill_(0.0)
            self.e_raw.fill_(-0.5)

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.shape[0], -1)

        polys = recursive_poly_basis(
            x, self.degree,
            self.a_raw, self.b_raw, self.c_raw, self.d_raw, self.e_raw,
            param_bound=self.param_bound
        )
        out = torch.einsum('bid,oid->bo', polys, self.weights)
        return out


class RecursivePolyKAN_FashionMNIST(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=64, num_classes=10, degree=3):
        super().__init__()
        self.flatten = nn.Flatten()

        self.kan1 = RecursivePolyKANLayer(input_dim, hidden_dim, degree=degree)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.drop1 = nn.Dropout(0.1)

        self.kan2 = RecursivePolyKANLayer(hidden_dim, hidden_dim, degree=degree)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.drop2 = nn.Dropout(0.1)

        self.kan3 = RecursivePolyKANLayer(hidden_dim, num_classes, degree=degree)

    def forward(self, x):
        x = self.flatten(x)
        x = self.kan1(x)
        x = self.norm1(x)
        x = self.drop1(x)

        x = self.kan2(x)
        x = self.norm2(x)
        x = self.drop2(x)

        x = self.kan3(x)
        return x


# =========================
# 2. Data loaders
# =========================

FASHION_MNIST_MEAN = (0.2860,)
FASHION_MNIST_STD = (0.3530,)

def make_fashion_mnist_loaders(batch_size=64, val_size=5000):
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(FASHION_MNIST_MEAN, FASHION_MNIST_STD)
    ])

    full_train_ds = datasets.FashionMNIST(
        './data_fashion_mnist',
        train=True,
        download=True,
        transform=tf
    )

    test_ds = datasets.FashionMNIST(
        './data_fashion_mnist',
        train=False,
        download=True,
        transform=tf
    )

    train_size = len(full_train_ds) - val_size
    train_ds, val_ds = random_split(
        full_train_ds,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=1000, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=1000, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader


# =========================
# 3. Training loop
# =========================

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            out = model(data)
            loss = criterion(out, target)
            pred = out.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
            loss_sum += loss.item()

    return loss_sum / len(loader), 100.0 * correct / total


def train_fashion_mnist(epochs=50, batch_size=64, device='cuda'):
    print("=" * 60)
    print("Recursive-Polynomial KAN on Fashion-MNIST")
    print(f"Device: {device} | Epochs: {epochs} | Batch: {batch_size}")
    print("=" * 60)

    train_loader, val_loader, test_loader = make_fashion_mnist_loaders(batch_size=batch_size)

    model = RecursivePolyKAN_FashionMNIST(
        input_dim=784, hidden_dim=64, num_classes=10, degree=3
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_test_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        start = time.time()
        epoch_loss = 0.0

        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        val_loss, val_acc = evaluate(model, val_loader, device)
        test_loss, test_acc = evaluate(model, test_loader, device)

        best_val_acc = max(best_val_acc, val_acc)
        best_test_acc = max(best_test_acc, test_acc)

        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - start

        with torch.no_grad():
            a = torch.tanh(model.kan1.a_raw).item()
            b = torch.tanh(model.kan1.b_raw).item()
            c = torch.tanh(model.kan1.c_raw).item()
            d = torch.tanh(model.kan1.d_raw).item()
            e = torch.tanh(model.kan1.e_raw).item()

        print(
            f"Epoch {epoch:2d}/{epochs} | "
            f"Train Loss: {avg_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | Best Val: {best_val_acc:.2f}% | "
            f"Test Acc: {test_acc:.2f}% | Best Test: {best_test_acc:.2f}% | "
            f"Time: {elapsed:.1f}s"
        )
        print(
            f"  Recurrence (tanh-scaled, layer1): "
            f"a={a:.3f}, b={b:.3f}, c={c:.3f}, d={d:.3f}, e={e:.3f}"
        )

        scheduler.step()

    return model


# =========================
# 4. Main
# =========================

if __name__ == '__main__':
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    EPOCHS = 50
    BATCH_SIZE = 64

    model = train_fashion_mnist(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        device=DEVICE
    )

    torch.save(model.state_dict(), 'recursive_poly_kan_fashion_mnist.pth')
    print("Model saved: recursive_poly_kan_fashion_mnist.pth")